In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

In [2]:
from Forecaster_Agent import ForecasterAgent

In [3]:
load_dotenv()

True

In [4]:
@tool
def generate_financial_forecast(days: int = 60) -> str:
    """
    Runs the BudAI mathematical forecaster to predict future bank balances.

    Args:
        days (int): The number of days into the future to forecast. 
    """
    try:
        agent = ForecasterAgent()
        real_balance = agent.fetch_live_balance()
        S0, mu, sigma = agent.fetch_and_calculate_parameters(real_balance, 60)
        agent.run_cpp_simulation(S0, mu, sigma, days=days, paths=1000)

        df = pd.read_csv("all_paths.csv", header=None)
        return (
            f"Forecast for {days} days:\n"
            f"- Careless: £{df.iloc[0].iloc[-1]:.2f}\n"
            f"- Expected: £{df.iloc[1].iloc[-1]:.2f}\n"
            f"- Optimal: £{df.iloc[2].iloc[-1]:.2f}"
        )
    except Exception as e:
        return f"Error: {str(e)}"

In [5]:
llm = ChatOllama(
    model="qwen3:8b",
    validate_model_on_init=True,
    temperature=0,
    base_url="http://localhost:11434"
).bind_tools([generate_financial_forecast])

In [ ]:
generate_financial_forecast.args_shcmea.model_json_schema()

{'description': 'Runs the BudAI mathematical forecaster to predict future bank balances.\n\nArgs:\n    days (int): The number of days into the future to forecast. ',
 'properties': {'days': {'default': 60, 'title': 'Days', 'type': 'integer'}},
 'title': 'generate_financial_forecast',
 'type': 'object'}

In [1]:
import torch
print(torch.__version__)

2.12.0.dev20260221


In [8]:
from cryptography.fernet import Fernet
import os

In [24]:
enc_key = os.getenv(
    "ENCRYPTION_KEY", b'cw_8H_1M4bX_3nF8vO5n3Y7A8xQ3_1m8aT2vP5_v5r8=')
cipher_suite = Fernet(enc_key)

In [25]:
enc_acc = "gAAAAABpt7FVKVHx7kwowHPlFC9oVBtNYcDn9J7FLPUeFbd7inYG3UpL0yoxLtMTns7i54U9p9uSQmbCLjKIL0VaMm1BpA-5ZxTqgepWcD_uEH2GiSQL-ycZtclejzK1Q90uBxN9qq8ungo-4TpPzGDOO_WIp4JOTmvzwrx143WV-QaHONpfXwctG4xzey8coKFNIqMQEhRI35opZzT9fkGjuarI8b88DsJCxUUWijQmMR9uHySzs7RsJ_Gwi1ddg-KktGmBaWUpvxSWDmf6Rq7AQ2BaLltSZF3E_V2MfhiWpsouy-p0NExsiNxJLqFs1DcSUD7uG8v3hf9QiTkQKbkbbtIp1jW6a4prKkdQG-88AoCOtZ3HZbOc_O_6B5yxqU94taoey989ct99pTJwY6fbCWXYS5UfXH8vRLO1JcJHEb51eLNFXw1FVNOvYP14Bta-4XLhONijm-DesXmH5EyFqeLXyadAXLQO7MsCA3qsL6TI1gGeq4C0rYDElDWzGlPEU75nBY28bVeO3di-9Q_CAsxID2j5zZnwVPva8C9msTptROJUhcNNIuyfb5FWsa8uQnQ9xbT6U0SRaE2t-gFdFFviF_r7d_Ab-s-kl7qRbzziiM_2of5Yb2yatV4ejZwsWT3HMrWhvB2GvEWAq6GcseAw7fCEG89gCM8fkP4QCpGm4RyfVbxbfBZSzd968N7ID_mbESJLyRXEgcbESL3n3UfHTQ_3wJwz4Czj4PuvjQ0940jSzBbUvxcwBH4xPS1tPiLUE4-Om4iXhePFy4qcJnsgYKLDt3NxSGjH47KmH1HBDTZjqd1CTQpPHLqlxrEUtVI7lbveeqv38_yA_FnIM2YpLNu4nrc8OzJepYLJIc9phXsfQqtS4aaNTboxil_PsZfKAzRibbZIwER9SBBHI2CSgmnEASsvpb7Cz_D4TsH8Z8JIlTMyh_T91HX59QMqJ9f9ZRcWOC0yzJQX2OXW1ahbvG4eAt7ajR8F1gBlEoVXmXLGuUoMwDONxFI4vB-LcG4t9p9omPLHQSAggrEj-tTpcXQlGVKqoWb8w_q2QVDPQNvM2RFVifhRvjlAWau6chkwlqCcCHHN9Z3jtQqbs8Zn_OTeuW926Iml5gPmi6T6LcVGAaCC5cXqewWCSmCoDC2KBwYyS-jtteDPT08J3IVs5eVb-VjkZdiq6Al_7dDy1N1lTIXsXg34UtA_c-Oe9wwsActNMVqRPifY2g6Q2VJUf65Sjwo-0CTTtW3_yC3DNsV_25BjBnDxLnXtHaXkEhjFYBQLZh9uPfhTKIP6c2G3l3LKYihKhKBrVJ7XtQige7dydjuUeiGnYlkxBCYt3Wj5YmGa_BbRlmv-wuPVZIKERbHthOr2e9VsMXSKhtu0hy8N5e2_6UwrXPwkAxJnJTtOi4S3urncFN5B8uaDjDnr-fgzlLYsunSHqwtJIh6Qih_-a1_sAo2H4ZDb9ZXmTB18HSok19wq5rRsOj85TDnwPsbUtN6SK_8YHQdJw3E8_oIf4lZX8r34zK8mv1iyEYgJu_N3awMR0f2w-__WxQk5jBggziBUc0OwTPyjxLhk2JVqGM8mwt19Pb4Yr5GTdkqvj3BB5G6qfuad56dRiN0XrwJWKylPQ-GhfkuwI37v-sdi-lxybxFznNsyWq3Su8V31pfLplttE4FTqPXX5gtyZ2Y_TQLdSnqWCwqfUBMlc92Omrybe3lKTalfsctv48d08s8zga7H4n7bBc3cZuU9dHWlkQXfHHvoLTNxHzPxC8tBBu7pDO_J0_ewShihIlUK-BuX_6OgXTeDGEHa6mclZY4WdzAaXG4F5q6YWWkJG7GVIE4QNgJoTvrmd_6aauR4XPT_5wd0cUTBL3kM8yfSor8t5NvCfAm0GErS2itWEG5Kq2p4ZIokdF5mMaf-Fh2s8l7elZW72hjesQDNZXdexCb8zzJ4JP7_qfPUOH710sic1QZ-HM2iE3GhejUvVR12Fqdg2IjgJyLayedOnE8pcrMvE1ZGJCYywlDERJEMq8s9t4KNB5pWdkyAQVM7f3SQB0H9jPBhM5KCbiR4eueQSwqJxa06UnOiiAXJLfFgjZdYeZcUg4535yKJZsHGrywPzF-cK2x7pR6AuBZGYYEoD-7QCykvs5p8ClrAsh4PoGktgZTXfHkKOinnT96OqaHc_VLNW7X8nNkzlZAp4X-c6jwAXf5xGm0yBiaOgbU7WHprTepgs9JfrAR74eo2O0WHARlPGk8NmQV9xJlXQiFb_jwSjBkPHe6reynch1bc8NwxNcCBgoa-Es_wTXYhQLxKCvpttqjuMZ88BnUfsR1e_xOzu2ccGiN-2wL58peoB1n1jGuV3ikoMQM85PXcuFKVa-pGc-MxDYTmPFqUqcI5s0Ba4vghgxU9UCplBWZzvtcjkZCmtTcghUKQk38IYUbiwfEStIxzz0JOxS-vS5uQDRq7SNyeiXVirlaU6682XSAL6mVTE5LRvj7QY05NhLKe3pJB3nLDWDoKrLyEaEBFRISWNw==		"

In [26]:
access_code = cipher_suite.decrypt(enc_acc).decode()

In [27]:
access_code

'eyJhbGciOiJSUzI1NiIsImtpZCI6IjE0NTk4OUIwNTdDOUMzMzg0MDc4MDBBOEJBNkNCOUZFQjMzRTk1MTBSUzI1NiIsIng1dCI6IkZGbUpzRmZKd3poQWVBQ291bXk1X3JNLWxSQSIsInR5cCI6ImF0K2p3dCJ9.eyJpc3MiOiJodHRwczovL2F1dGgudHJ1ZWxheWVyLmNvbSIsIm5iZiI6MTc3MzY0NjE2NSwiaWF0IjoxNzczNjQ2MTY1LCJleHAiOjE3NzM2NDk3NjUsImF1ZCI6ImRhdGFfYXBpIiwic2NvcGUiOlsiaW5mbyIsImFjY291bnRzIiwiY2FyZHMiLCJzdGFuZGluZ19vcmRlcnMiLCJkaXJlY3RfZGViaXRzIiwiYmFsYW5jZSIsInRyYW5zYWN0aW9ucyIsIm9mZmxpbmVfYWNjZXNzIl0sImFtciI6WyJwd2QiXSwiY2xpZW50X2lkIjoiZXhhbXBsZTEtYmYwMDNkIiwic3ViIjoiclVTUkZUcUVMOHI1YXRvcFlDN2JLTDU2b3F6cUFFZ0pCMzAxYjJ6WFBuZz0iLCJhdXRoX3RpbWUiOjE3NzM1NjcyMTMsImlkcCI6ImxvY2FsIiwic2lkIjoiYWZzcy1oTExrTzNiWXZobFJldjB1aG1La3RHbkF6dWN5TjVxUXJOTDhrNXVjcjZnIiwiY29ubmVjdG9yX2lkIjoib2ItbGxveWRzIiwiY3JlZGVudGlhbHNfa2V5IjoiMDZkMzhlZTJhMDZiOWEzZjQxNGUyZGVlOWE1MTBlY2FkYTdhNzU5OWUwNzY1NzViMzA0MWE0NDgxNzg0NmYyNiIsInByaXZhY3lfcG9saWN5IjoiU2VwMjAyMyIsImNvbnNlbnRfaWQiOiI5YjUwZDcwZC0xMjA3LTRlZjYtOWJiMi1mOTI4NGM3OWQwZjMiLCJjb25zZW50X2V4cGlyeV90aW1lIjoiMTc4MTM0Mz

In [28]:
import requests

url = "https://api.truelayer.com/data/v1/accounts/3e9cb602bfa8b5b39e32c27c8c7c34e4/transactions?to=2025-01-01&from=2023-01-01"

headers = {
    "accept": "application/json",
    "authorization": f"Bearer {access_code}"
}

response = requests.get(url, headers=headers)

print(response.json())

{'error_description': 'SCA exemption has expired. This resource is protected and should be accessed shortly after PSU Authentication. In order to access this resource, please have the PSU re-authenticate.', 'error': 'sca_exceeded', 'error_details': {'provider_details': '403 access_denied: SCA exemption has expired. This resource is protected and should be accessed within 5 minutes of PSU Authentication. In order to access this resource, please have the PSU re-authenticate.'}}


In [29]:
response.json()

{'error_description': 'SCA exemption has expired. This resource is protected and should be accessed shortly after PSU Authentication. In order to access this resource, please have the PSU re-authenticate.',
 'error': 'sca_exceeded',
 'error_details': {'provider_details': '403 access_denied: SCA exemption has expired. This resource is protected and should be accessed within 5 minutes of PSU Authentication. In order to access this resource, please have the PSU re-authenticate.'}}

In [30]:
enc_ref = "gAAAAABpt7FVHmBay7Wabs20Ray8wZ_nITgDz8PhKH3szKd_sOlmaeHJ_eNvvP_OzrU9Ool2VuHq9kZ3Yx04EBqhQQOJfpRfgJBA2yw7a8ClWKyOW_hVKxHKhqucvctxdqcO7hyks0yJvE6qWFjn44bNeDlcBA_1GylkzMWCVugxltbiWedYEcY="
refresh_token = cipher_suite.decrypt(enc_ref).decode()
refresh_token

'CB487397A511420E2D58F1031526226432589FC6957B50E5CCBD5E49CF8E69FE'

In [31]:
url = "https://api.truelayer.com/data/v1/connections/extend"

payload = {
    "user_has_reconfirmed_consent": True,
    "user": {
        "id": "4a63b83c-6846-4ea3-a60c-bc2239bdc82e",
        "name": "Arnav Mishra",
        "email": "arnavpragya04@gmail.com",
    },
    "client_id": "example1-bf003d",
    "client_secret": "78ff19e9-c4c8-4fda-a470-7a720e8632c1",
    "refresh_token": refresh_token,
    "redirect_uri": "http://localhost:8080/callback"
}
headers = {
    "accept": "application/json",
    "content-type": "application/json",
    "authorization": f"Bearer {access_code}"
}

response = requests.post(url, json=payload, headers=headers)


In [32]:
response

<Response [404]>